# CatBoost 전용 — v4-1/v4-2 추천 (Colab GPU)

CPU에서 CatBoost 다중분류가 너무 느려, **CatBoost만 Colab GPU(`task_type='GPU'`)** 로 빠르게 학습한다.
- cat(다음 카테고리)·item(다음 아이템) 각각 CatBoost
- 베이지안 전처리탐색 + early stopping, 지표 top-1/top-5/MRR
- 결과물: `prep_CatBoost_rec.joblib`(+train parquet·bayes.json·first30) → zip 다운로드

**런타임 → 런타임 유형 변경 → GPU(T4)** 필수. 입력: `train_cat,test_cat,train_item,test_item.parquet` 업로드.


## 0) 설치 · 임포트 · GPU 확인


In [ ]:
!pip -q install optuna catboost pyarrow scikit-learn 2>/dev/null
import os, json, time, shutil, numpy as np, pandas as pd, joblib
import optuna; from optuna.samplers import TPESampler
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler, LabelEncoder
from catboost import CatBoostClassifier
optuna.logging.set_verbosity(optuna.logging.WARNING); SEED = 42; np.random.seed(SEED)
# GPU 확인
import subprocess
try: print(subprocess.check_output(['nvidia-smi','--query-gpu=name','--format=csv,noheader']).decode().strip(), '→ GPU OK')
except Exception: print('⚠ GPU 없음 — 런타임 유형을 GPU로 바꾸세요(CPU면 느립니다)')


## 1) 데이터 업로드 (4개 parquet)


In [ ]:
DATA='/content/recdata'; OUT='/content/cb_output'; os.makedirs(DATA,exist_ok=True); os.makedirs(OUT,exist_ok=True)
need=['train_cat.parquet','test_cat.parquet','train_item.parquet','test_item.parquet']
if [f for f in need if not os.path.exists(f'{DATA}/{f}')]:
    from google.colab import files; up=files.upload()
    for k,v in up.items(): open(f'{DATA}/{os.path.basename(k)}','wb').write(v)
print('ready:', [f for f in need if os.path.exists(f'{DATA}/{f}')])


## 2) 공통 전처리


In [ ]:
FEAT=['recency_days','tenure_days','ndays','n_events','n_view','n_cart','n_remove_from_cart','n_purchase','avg_price','purch_amt',
      'min_price','max_price','std_price','purchase_avg_price','remove_ratio','cart_purchase_ratio','n_categories','cat_entropy',
      'n_brands','brand_loyalty','n_sessions','events_per_session']
COUNTS=['ndays','n_events','n_view','n_cart','n_remove_from_cart','n_purchase','purch_amt','n_categories','n_brands','n_sessions']
COUNT_IDX=[FEAT.index(c) for c in COUNTS]
DSETS={'cat':dict(train='train_cat.parquet',test='test_cat.parquet',y='y_next_category'),
       'item':dict(train='train_item.parquet',test='test_item.parquet',y='y_next_item')}
MIN_COUNT=10
def load_ds(key):
    ds=DSETS[key]; tr=pd.read_parquet(f'{DATA}/{ds["train"]}'); te=pd.read_parquet(f'{DATA}/{ds["test"]}')
    vc=tr[ds['y']].value_counts(); keep=set(vc[vc>=MIN_COUNT].index)
    tr=tr[tr[ds['y']].isin(keep)]; te=te[te[ds['y']].isin(keep)]
    le=LabelEncoder().fit(tr[ds['y']].values)
    Xtr=np.nan_to_num(tr[FEAT].values.astype(float)); Xte=np.nan_to_num(te[FEAT].values.astype(float))
    return dict(tr=tr,te=te,Xtr=Xtr,Xte=Xte,ytr=le.transform(tr[ds['y']].values),yte=le.transform(te[ds['y']].values),
                ncls=len(le.classes_),classes=le.classes_.tolist(),y=ds['y'])
def transform(Xfit,Xapply,prep):
    a,b=Xfit.copy(),Xapply.copy()
    if prep['log_counts']:
        a[:,COUNT_IDX]=np.log1p(np.clip(a[:,COUNT_IDX],0,None)); b[:,COUNT_IDX]=np.log1p(np.clip(b[:,COUNT_IDX],0,None))
    sc={'standard':StandardScaler(),'minmax':MinMaxScaler(),'robust':RobustScaler()}.get(prep['scaler'])
    if sc is not None: sc.fit(a); a,b=sc.transform(a),sc.transform(b)
    return a,b,sc
def topk(m,X,y,k=5):
    p=m.predict_proba(X); cls=np.asarray(m.classes_); kk=min(k,p.shape[1])
    idx=np.argpartition(-p,kth=kk-1,axis=1)[:,:kk]; top=cls[idx]
    return float(np.mean([y[i] in top[i] for i in range(len(y))]))
def mrr(m,X,y):
    p=m.predict_proba(X); cls=np.asarray(m.classes_); o=np.argsort(-p,axis=1); rr=0.0
    for i,yt in enumerate(y):
        r=np.where(cls[o[i]]==yt)[0]
        if len(r): rr+=1.0/(r[0]+1)
    return float(rr/len(y))
print('prep ready')


## 3) CatBoost(GPU) 베이지안 + ES + 저장


In [ ]:
N_ITER=400; ES=30   # GPU라 넉넉히(ES가 자동 단축)
def run_catboost(key, n_trials=8):
    d=load_ds(key); ncls=d['ncls']; t0=time.time()
    Xh,Xv,yh,yv=train_test_split(d['Xtr'],d['ytr'],test_size=0.2,random_state=SEED,stratify=d['ytr'])
    def mk(hp,cw):
        return CatBoostClassifier(iterations=N_ITER,depth=hp['depth'],learning_rate=hp['lr'],l2_leaf_reg=hp['l2_leaf_reg'],
                                  loss_function='MultiClass',od_type='Iter',od_wait=ES,auto_class_weights=('Balanced' if cw else None),
                                  task_type='GPU',devices='0',random_state=SEED,verbose=0)
    def fit(m,Xt,yt,Xvv,yvv):
        seen=np.isin(yvv,np.unique(yt)); m.fit(Xt,yt,eval_set=(Xvv[seen],yvv[seen]),verbose=0); return m
    def obj(t):
        prep={'scaler':t.suggest_categorical('scaler',['none','standard','robust']),'log_counts':t.suggest_categorical('log_counts',[True,False]),'imbalance':t.suggest_categorical('imbalance',['none','classweight'])}
        hp={'depth':t.suggest_int('depth',4,8),'lr':t.suggest_float('lr',0.03,0.3,log=True),'l2_leaf_reg':t.suggest_float('l2_leaf_reg',1.0,10.0)}
        cw='balanced' if prep['imbalance']=='classweight' else None
        Xa,Xb,_=transform(Xh,Xv,prep); m=fit(mk(hp,cw),Xa,yh,Xb,yv); return topk(m,Xb,yv,5)
    st=optuna.create_study(direction='maximize',sampler=TPESampler(seed=SEED)); st.optimize(obj,n_trials=n_trials)
    bp=st.best_params; prep={k:bp[k] for k in ('scaler','log_counts','imbalance')}; hp={k:bp[k] for k in bp if k not in prep}
    cw='balanced' if prep['imbalance']=='classweight' else None
    Xfa,Xfb,scaler=transform(d['Xtr'],d['Xte'],prep)
    Xh2,Xv2,yh2,yv2=train_test_split(Xfa,d['ytr'],test_size=0.1,random_state=SEED,stratify=d['ytr'])
    clf=fit(mk(hp,cw),Xh2,yh2,Xv2,yv2)
    metrics=dict(oot_top1=round(topk(clf,Xfb,d['yte'],1),4),oot_top5=round(topk(clf,Xfb,d['yte'],5),4),oot_mrr=round(mrr(clf,Xfb,d['yte']),4),n_classes=ncls,n_features=len(FEAT))
    mo=f'{OUT}/{key}/CatBoost'; os.makedirs(mo,exist_ok=True)
    joblib.dump({'model_name':f'CatBoost_rec_{key}','model_type':'tree','task':'multiclass_recommendation','target':d['y'],
                 'feature_order':FEAT,'prep':prep,'hp':hp,'scaler':scaler,'classifier':clf,'classes':d['classes'],'metrics':metrics}, f'{mo}/prep_CatBoost_rec.joblib')
    pd.DataFrame(Xfa,columns=FEAT).assign(**{d['y']:d['ytr'],'user_id':d['tr']['user_id'].values}).to_parquet(f'{mo}/CatBoost_rec_train.parquet',index=False)
    json.dump({'best_params':bp,'metrics':metrics,'n_trials':n_trials}, open(f'{mo}/CatBoost_rec_bayes.json','w',encoding='utf-8'), ensure_ascii=False, indent=2)
    with open(f'{mo}/CatBoost_first30.txt','w',encoding='utf-8') as f:
        f.write(f'# CatBoost({key}) top1 {metrics["oot_top1"]} top5 {metrics["oot_top5"]} MRR {metrics["oot_mrr"]} | {prep}\n\n')
        f.write(d['tr'][['user_id']+FEAT+[d['y']]].head(30).to_string(index=False))
    r=dict(dataset=key,model='CatBoost',**metrics,prep=f"{prep['scaler']}|{prep['imbalance']}",sec=round(time.time()-t0)); print(r); return r
print('run_catboost ready')


## 4) 실행(cat·item) + zip 다운로드


In [ ]:
res=[run_catboost('cat', 8), run_catboost('item', 8)]
pd.DataFrame(res).to_csv(f'{OUT}/catboost_results.csv', index=False); print(pd.DataFrame(res))
zip_path=shutil.make_archive('/content/catboost_rec_output','zip',OUT); print('ZIP', zip_path, round(os.path.getsize(zip_path)/1e6,1),'MB')
from google.colab import files; files.download(zip_path)
